# LAST TIDE — Limpieza y análisis del Global Shark Attack FileProyecto final de *data wrangling* · Data Analytics · Ironhack**Equipo:** Adriana Rojas Rodríguez · Edvaldo · Jordi Callejo · Alejandro · Jose GalindoEl GSAF describe incidentes **documentados**. No mide exposición ni riesgo individual.Todo lo que hay en este cuaderno es descriptivo.

## 0 · Preparación

In [ ]:
import sys, ossys.path.append(os.path.abspath(".."))import pandas as pdimport numpy as npfrom src.cleaning import (    run_pipeline, load_raw,    subset_without_invalid, subset_with_outcome,    N_FILAS_ORIGINALES, N_SIN_INVALID, N_CON_DESENLACE,)from src.analysis import (    activity_frequency, activity_by_outcome, outcome_by_era,    geography_concentration, species_share,    species_coverage_by_outcome, cramers_v,)RUTA = "../data/GSAF5.xls"pd.set_option("display.max_columns", 50)

## 1 · Punto de partidaAntes de tocar nada, miramos qué hay. Debe dar **7.125 filas × 23 columnas**.

In [ ]:
raw = load_raw(RUTA)print(raw.shape)raw.head()

In [ ]:
# Qué columnas trae y cuánto falta en cada una.# OJO: varias columnas tienen un espacio al final del nombre.print([repr(c) for c in raw.columns])raw.isna().mean().sort_values(ascending=False).head(12)

## 2 · Limpieza`run_pipeline()` encadena todas las funciones de `src/cleaning.py` en orden.Cada una está documentada con la cifra de control que debe salir.> **TODO:** implementar las funciones de `cleaning.py` antes de ejecutar esta celda.

In [ ]:
df = run_pipeline(RUTA)df.shape

In [ ]:
# Control 1 — el desenlace debe repartirse así:#   N = 4.986 · Y = 1.496 · UNKNOWN = 643   (suma 7.125)df["fatal_clean"].value_counts(dropna=False)

## 3 · Las tres basesEl error más fácil de este dataset es comparar cifras calculadas sobreuniversos distintos. Aquí se fijan y se comprueban.

In [ ]:
sin_invalid = subset_without_invalid(df)   # base de H4con_desenlace = subset_with_outcome(df)      # base de H3print(f"base completa      {len(df):>6,}")print(f"sin Invalid        {len(sin_invalid):>6,}")print(f"con desenlace      {len(con_desenlace):>6,}")assert len(df) == N_FILAS_ORIGINALESassert len(sin_invalid) == N_SIN_INVALIDassert len(con_desenlace) == N_CON_DESENLACEprint("\nLas tres bases cuadran.")

## 4 · H1 — ¿Qué actividades aparecen?Base: 7.125. Control: surf 1.162 (16,31 %), natación 1.076 (15,10 %).

In [ ]:
activity_frequency(df)

## 5 · H2 — ¿Dónde se concentran?Control: 66,9 % en tres países · 4.569 lugares distintos.

In [ ]:
geography_concentration(df)

## 6 · H3 — ¿Qué cambia entre desenlaces?Base: 6.482 (N = 4.986, Y = 1.496). Las columnas suman 100 % **dentro de cadadesenlace**: comparamos composición, no probabilidad de morir.

In [ ]:
tabla = activity_by_outcome(df)tabla

In [ ]:
# Tamaño del efecto. Con miles de filas el p-valor sale significativo casi# siempre, así que lo que se presenta es la V de Cramér. Control: ~0,385contingencia = pd.crosstab(con_desenlace["activity_group"], con_desenlace["fatal_clean"])print(f"V de Cramér = {cramers_v(contingencia):.3f}")

In [ ]:
# Sensibilidad por época: cuánto de la diferencia es histórico.# Control: natación cae de +13,6 a +5,6 puntos desde 2000; surf se mantiene.outcome_by_era(df, corte=2000)

## 7 · H4 — ¿Qué especies se identifican?Base: 6.573 sin Invalid. Solo 2.058 tienen especie (31,3 %).

In [ ]:
species_share(df)

In [ ]:
# PENDIENTE DE CERRAR: la cobertura de especie por desenlace.# En el equipo circulan dos cifras incompatibles (18,6 % y 31,7 % para los# mortales) y ninguna cuadra con las 2.058 identificaciones. Esta celda lo# resuelve: la cifra que salga aquí es la buena.species_coverage_by_outcome(df)

## 8 · Conclusiones| | Resultado | Tipo de apoyo ||---|---|---|| **H1** | Reparto desigual por actividad; no mide riesgo | Descriptivo || **H2** | Concentración por país, dispersión por lugar | Descriptivo || **H3** | Composición distinta por desenlace; depende de la época | Descriptivo + χ² exploratorio || **H4** | Especies limitadas por la falta de identificación | Descriptivo |Ninguna hipótesis queda «demostrada»: hay **apoyo observacional, sincausalidad**. Convertir estos hallazgos en mecánicas de LAST TIDE es unadecisión creativa del equipo, no una conclusión del GSAF.